# g_patent
## Overview of the Table

In [1]:
# load libraries 
import pandas as pd 
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import numpy as np
import pyarrow.parquet as pypq
import textwrap 
from time import time 
import plotly.io as pio

tqdm.pandas()
plt.rcParams.update({'font.size': 22})
sns.set(style="ticks", context="talk")
plt.style.use("dark_background")
pd.options.plotting.backend = 'plotly'
pio.templates.default = 'plotly_dark+presentation'

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [2]:
# some helper functions 
def read_parquet(path, engine='pyarrow', columns=None, convert_dtypes=True, **args):
    """
    Read a parquet file (or a directory of parquet files) 
    columns: list of columns to read, by default, read all columns
    convert_dtypes: if True, convert datatypes to save RAM (takes extra time)
    """
    name = path.stem 
    column_st = 'columns="all"' if columns is None else f'{columns=!r}'
    print(f'\nReading {column_st} from {path!r} using {engine=!r}.')

    tic = time()
    df = pd.read_parquet(path, engine=engine, columns=columns, **args)
    toc = time()
    print(f'Read {len(df):,} rows from {path.stem!r} in {toc-tic:.2f} sec.')
    
    if convert_dtypes:
        tic = time()
        size_before = df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024

        string_cols_d = {}
        for col, dtype in df.dtypes.to_dict().items():
            if dtype == 'object':  # convert object columns to string
                string_cols_d[col] = 'string[python]'
            if col == 'type' or col == 'concept_name':
                if dtype != 'category':
                    string_cols_d[col] = 'category'
            if col == 'publication_month':
                if dtype != 'uint8':
                    string_cols_d[col] = 'uint8'
            if col == 'score':
                if dtype != 'float16':
                    string_cols_d[col] = 'float16'
        # print(f'{string_cols_d=}')
        df = df.astype(string_cols_d) 
        
        size_after = df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024
        toc = time()
        print(f'Converting dtypes took {toc-tic:.2f} sec. Size before: {size_before:.2f}GB, after: {size_after:.2f}GB.')
    
    display('Top 3 rows:', df.head(3))
    return df


def peek_parquet(path):
    """
    peeks at a parquet file (or a directory containing parquet files) without reading the whole thing and prints the following:
    * Path
    * schema
    * number of pieces (fragments)
    * number of rows 
    """
    parq_file = pypq.ParquetDataset(path)
    piece_count = len(parq_file.fragments)
    schema = textwrap.indent(parq_file.schema.to_string(), ' '*4)
    row_count = sum(frag.count_rows() for frag in parq_file.fragments)
    if Path(path).is_dir():
      size = sum(Path(frag.path).stat().st_size for frag in parq_file.fragments)
    else:
      size = path.stat().st_size
    
    st = [
        f'Name: {path.stem!r}',  
        f'Path: {str(path)!r}',
        f'Size: {size/1024/1024/1024:.2g} GB',
        f'Files: {piece_count:,}',
        f'Rows: {row_count:,}',
        f'Schema:\n{schema}',
        f'5 random rows:',
    ]
    print('\n'.join(st))
    sample_df = parq_file.fragments[0].head(5).to_pandas()  # read 5 rows from the first fragment
    display(sample_df)

    return

def read_smaller_tables(name):
    """
    Some smaller tables exist as a CSV only
    """
    assert name in ['institutions', 'institutions_geo', 'concepts']
    path = basepath / 'csv-files'/ month / name
    df = pd.read_csv(f'{path}.csv.gz', engine='c')
    return df

def tsv_to_parquet(tsv_path, output_filename=None, dtype=None, **read_csv_kwargs):
    """
    Convert a TSV file to Parquet format and save it in the current directory.
     
    Args:
        tsv_path: Path to the TSV file to convert
        output_filename: Optional output filename (defaults to same name as TSV but .parquet)
        dtype: Optional dict of column dtypes for reading TSV
        **read_csv_kwargs: Additional arguments to pass to pd.read_csv
    
    Returns:
        Path to the created parquet file
    """

    #get the output path
    notebook_dir = Path.cwd()
    if output_filename is None:
        output_filename = Path(tsv_path).stem + '.parquet'
    output_path = notebook_dir / output_filename

    if not output_path.exists():
        print(f"Converting {tsv_path} to Parquet...")
        df = pd.read_csv(
            tsv_path,
            sep = '\t',
            dtype=dtype,
            low_memory=False, 
            **read_csv_kwargs
        )
        df.to_parquet(output_path, index=False)
    return output_path
            

In [ ]:
# Convert TSV to Parquet (only runs if parquet doesn't exist yet)
output_path = tsv_to_parquet(
    '/data/shared/USPTO-patents/g_patent.tsv',
    dtype={'patent_id': str}
)

# Peek at the parquet file
peek_parquet(output_path)

# Read it using your helper function
g_patents_df = read_parquet(output_path)

Converting /data/shared/USPTO-patents/g_patent.tsv t Parquet...
Name: 'g_patent'
Path: '/home/jupyter-mgarciamelo/patentDataDiscovery/gPatent/g_patent.parquet'
Size: 0.32 GB
Files: 1
Rows: 9,361,444
Schema:
    patent_id: string
    patent_type: string
    patent_date: string
    patent_title: string
    wipo_kind: string
    num_claims: int64
    withdrawn: int64
    filename: string
    -- schema metadata --
    pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1027
5 random rows:


,patent_id,patent_type,patent_date,patent_title,wipo_kind,num_claims,withdrawn,filename
0,10000000,utility,2018-06-19,Coherent LADAR using intra-pixel quadrature de...,B2,20,0,ipg180619.xml
1,10000001,utility,2018-06-19,Injection molding machine and mold thickness c...,B2,12,0,ipg180619.xml
2,10000002,utility,2018-06-19,Method for manufacturing polymer film and co-e...,B2,9,0,ipg180619.xml
3,10000003,utility,2018-06-19,Method for producing a container from a thermo...,B2,18,0,ipg180619.xml
4,10000004,utility,2018-06-19,"Process of obtaining a double-oriented film, c...",B2,6,0,ipg180619.xml



Reading columns="all" from PosixPath('/home/jupyter-mgarciamelo/patentDataDiscovery/gPatent/g_patent.parquet') using engine='pyarrow'.
